In [ ]:
import psutil
nthreads = psutil.cpu_count(logical=False)
# nthreads = 1

In [ ]:
import os,sys
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [ ]:
import sys, os
sys.path.append('../')
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark
import energy
import sparse_matrices, differential_operators

import numpy as np
import time, copy
import igl
import param_utils, initial_utils, extra_utils, sim_utils

In [ ]:
import matplotlib
from matplotlib import pyplot as plt

# Load Mesh (rest and deformed)

In [ ]:
model_rest = 'pants_rest.obj'
model_deformed = 'pants_deformed.obj'

In [ ]:
m_rest = mesh.Mesh(f'ToysMesh/{model_rest}', embeddingDimension=3)
m_deformed = param_utils.load(f'ToysMesh/{model_deformed}')

In [ ]:
print(f"Model: {model_rest} Vertices: {m_rest.numVertices()}")
print(f"Model: {model_rest} Elements: {m_rest.numElements()}")

# Set Initial UV using deformed mesh

In [ ]:
uv = mesh_energy.NodalVars(m_rest, 2)
uv_init = m_deformed.vertices()
uv.setVars(uv_init.ravel())

In [ ]:
e = energy.SymmetricDirichlet(2)
param = mesh_energy.Parametrization(m_rest, uv, e)
objectives = [param]

prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)

In [ ]:
uvs = {}
uvs['initial_bend'] = uv_init

In [ ]:
param_utils.analysisPlots(m_rest, uvs)

# Prob and Opt Setting

In [ ]:
import flip_avoiding_step_length
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m_rest.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8

In [ ]:
# Nullspace pinning strategy
# TODO: add epsilon * low rank.
FIX_VARS = True
if FIX_VARS:
    fv = sim_utils.getBBoxVars(m_rest, sim_utils.BBoxFace.MIN_X, dimension=2)
    prob.setFixedVars(fv)
else:
    # prob.hessianShift = 1e-5
    nf.elementHessianShift = 1e-6

In [ ]:
fv

In [ ]:
# Work around energy nullspace by adding a small shift
# prob.hessianShift = 1e-10
prob.useRelativeHessianShift = True
param.useXBasedProjection = False
opt = prob.optimizer()
opt.options.niter = 500
opt.options.gradTol = 2e-8

opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2
opt.options.hessianProjectionController.startWithProjectionActive = False

In [ ]:
prob.energy()

# Newton Optimier

In [ ]:
opt.options.niter = 5
prob.setVars(uv_init.ravel())
opt.optimize()

In [ ]:
uv_cur = prob.getVars().reshape(-1,2)

# Extrapolation

In [ ]:
step = opt.newton_step()

In [ ]:
alpha = 1.0

## Construct Factorizer L

In [ ]:
L_matrix = differential_operators.laplacian(m_rest, upperTriOnly=True)
L_sparse = sparse_matrices.SuiteSparseMatrix(L_matrix)
L_sparse.rowColRemoval([0])
L_sparse.symmetry_mode = L_sparse.symmetry_mode.UPPER_TRIANGLE

Linv = sparse_matrices.CholeskyFactorizer()
Linv.factorize(L_sparse)

# Alphas and Energies

In [ ]:
alpha = 1

In [ ]:
uv_new_euler, F_extra_euler, d_grad_euler = extra_utils.paramNewtonstepExtrapolation(m_rest, step, param, alpha, Linv, 
                                                                                        fixedVind=90, fixedUV=uv_init[90])

In [ ]:
d_grad_euler.shape

In [ ]:
brek

In [ ]:
from curved_linesearch import visualization

In [ ]:
new_vector_field = uv_new_euler - uv_cur

In [ ]:
F_extra_euler.shape

In [ ]:
m = m_rest
visualization.plot_mesh(uv_cur, m_rest.elements())
# visualization.plot_element_labels(m_rest, uv_cur)
visualization.plot_vector_field(uv_cur, new_vector_field, ax=plt.gca())

In [ ]:
visualization.plot_mesh(uv_cur, m_rest.elements())
visualization.plot_vector_field(uv_cur, step.reshape(-1,2), ax=plt.gca())

In [ ]:
np.linalg.norm(step.reshape(-1,2) - new_vector_field)

In [ ]:
brek

## Energy Plot

In [ ]:
alphas = np.linspace(0, 3.0, 100)

In [ ]:
prob.setVars(uv_cur.ravel())
uvs_extra = []
F_extra_list = []
d_grad_list = []
for a in alphas:
    uv_new, F_extra, d_grad = extra_utils.paramNewtonstepExtrapolation(m_rest, step, param, a, Linv)
    uvs_extra.append(uv_new)
    F_extra_list.append(F_extra)
    d_grad_list.append(d_grad)
uvs_extra_arr = np.array(uvs_extra)

In [ ]:
# Energy visualization
emin = np.inf
energies = []
linear_energies = []
gnorms = []
for uv_arr in uvs_extra_arr:
    prob.setVars(uv_arr.ravel())
    energies.append(prob.energy())
    gnorms.append(np.linalg.norm(prob.gradient()))
    
for a in alphas:
    prob.setVars(uv_cur.ravel() + a * step)
    linear_energies.append(prob.energy())

emin = min(emin, min(energies))
plt.plot(alphas, energies, label='Eulerian')
plt.plot(alphas, linear_energies, label='Linear')

plt.ylim(energies[0] - (energies[0] - emin) * 1.05, energies[0] + (energies[0] - emin) * 1.05)
# plt.legend(loc='center left', bbox_to_anchor=[1.0, 0.5])
plt.xlabel('Line Search Parameter ⍺')
plt.ylabel('Energy')
plt.legend()
plt.tight_layout()

In [ ]:
# vector_field_for_uv = np.array([uvs_extra_arr[i] - uvs_extra_arr[0] for i in range(1, 100)])

# Viewer

In [ ]:
v_mesh = viewer.Viewer(m_deformed, wireframe=True)
v_mesh.show()

In [ ]:
m_deformed.vertices()

In [ ]:
brek